# The field effect: what it is, and what explains it (so far: nothing)

This is the central research object of this repository, and the groundwork for E06
(`docs/superpowers/plans/2026-08-29-E06-does-soil-explain-the-field-effect.md`). Reads
`../data/training_set.parquet` only.

## 1. Setup

In [1]:
import pandas as pd, numpy as np
from argotech.lab.eval.variance import decompose, icc
from argotech.lab.estimand.targets import alpha_hat

panel = pd.read_parquet('../data/training_set.parquet')
panel.shape

(4596, 35)

## 2. The variance decomposition

`forward_z` decomposes as a time-invariant field effect plus a cohort-date effect plus residual.
Peer standardisation removes the cohort-date effect (that is what makes `forward_z` a z-score); it
was never designed to remove the field effect, and it does not. This is the reproduction gate for
this whole line of work -- it must print exactly `0.345 [0.2357, 0.4353] 0.0`.

The ICC / variance-components estimator is standard method-of-moments (Searle, Casella & McCulloch) — **[UNVERIFIED]**, see [`docs/CONCEPTS.md` § ICC and variance components](../docs/CONCEPTS.md#icc-and-variance-components); the 95% interval is a site-level bootstrap (Efron, 1979) — **[UNVERIFIED]**, see [`docs/CONCEPTS.md` § Bootstrap confidence intervals](../docs/CONCEPTS.md#bootstrap-confidence-intervals). **The decomposition itself — the 0.345 field-effect ICC and the exact-zero cluster ICC — is a measurement from this repository, not borrowed science**; its warrant is reproducibility (`experiments/E01_variance_decomposition.out`), not a citation, per [`docs/CONCEPTS.md`'s opening section](../docs/CONCEPTS.md#on-warrants-and-what-no-source-means).

In [2]:
r = decompose(panel)
print(round(r['site']['icc'], 4), [round(x, 4) for x in r['icc_ci']], round(r['cluster']['icc'], 4))
print()
print(f"field-effect (site) ICC : {r['site']['icc']:.3f}   95% CI {tuple(round(x,4) for x in r['icc_ci'])}")
print(f"cluster ICC             : {r['cluster']['icc']:.4f}")

0.345 [0.2357, 0.4353] 0.0

field-effect (site) ICC : 0.345   95% CI (0.2357, 0.4353)
cluster ICC             : 0.0000


## 3. The field effect is entirely within-cluster

If the field effect were a *regional* signature, blocking by cluster (leave-one-cluster-out) would
be controlling for it directly, and climate would be a live candidate. It is not: the between-cluster
share of the field effect itself is 0.0000.

The unit here is the **site** (n=122), not the row -- this is a question about a per-site constant.
The field effect is estimated the way the rest of this repository estimates it:
`lab.estimand.targets.alpha_hat`, a strictly-prior expanding mean of `ndvi_z_peer` with
`min_history=5`, averaged over each site's own valid rows.

In [3]:
a = alpha_hat(panel, column='ndvi_z_peer', unit='site_id', min_history=5)
site_effect = (panel.assign(alpha=a).dropna(subset=['alpha'])
               .groupby(['site_id','cluster']).alpha.mean().reset_index())
print(f"sites with an estimable field effect: {len(site_effect)}")
print(f"sd of the field effect across sites : {site_effect.alpha.std():.4f}")

r_site = icc(site_effect.alpha, site_effect.cluster)
print(f"between-cluster ICC of the per-site effect: {r_site['icc']:.4f}")
print(f"within-cluster sd (sqrt of the ICC decomposition's within-group variance): "
      f"{np.sqrt(r_site['s2_within']):.4f}")
print()
print("cluster means:")
print(site_effect.groupby('cluster').alpha.mean().round(3))

sites with an estimable field effect: 122
sd of the field effect across sites : 0.5086
between-cluster ICC of the per-site effect: 0.0000
within-cluster sd (sqrt of the ICC decomposition's within-group variance): 0.5135

cluster means:
cluster
Benue_River_Basin      0.015
Kaduna_Grain_Belt      0.028
Kano_Sudan_Savannah    0.063
Kenya_Rift_Valley     -0.040
Name: alpha, dtype: float64


**Reading this table.** Weather/climate barely varies within a 120–180 km cluster, and
`forward_z` already cancels what the peer cohort shares at each date -- so climate cannot be the
within-cluster field effect. Static soil (texture, organic carbon, pH, drainage) varies strongly at
field scale *and* is time-invariant, which is what a time-invariant field effect requires. Soil
*moisture* (SMAP) varies at field scale too, but it is dynamic, not time-invariant -- the wrong
instrument for a constant. This is the E06 hypothesis: soil is the candidate; climate is ruled out
by the numbers above, not by assumption.

## 4. What the panel already explains: elevation, latitude, longitude

The correlations below are computed after **demeaning both the field effect and each covariate by cluster** — the within/demeaning transformation of Frisch, R., Waugh, F.V. (1933), *Econometrica* 1(4):387–401, and Lovell, M.C. (1963), *JASA* 58(304):993–1010 — **[UNVERIFIED]**, see [`docs/CONCEPTS.md` § The within transformation, and Frisch–Waugh–Lovell](../docs/CONCEPTS.md#the-within-transformation-and-frischwaughlovell). The same cross-sectional demeaning applied specifically to crop-yield anomalies: *Forecasting Crop Yield Anomalies on Panel Data via Spatially Demeaned Ensembles*, J. Agric. Biol. Environ. Stat. (2026), [https://doi.org/10.1007/s13253-026-00743-8](https://doi.org/10.1007/s13253-026-00743-8) — cited by title and DOI deliberately; **authorship [UNVERIFIED]**, and it must stay that way.

In [4]:
site = (panel.assign(alpha=a).dropna(subset=['alpha'])
        .groupby(['site_id','cluster'])
        .agg(alpha=('alpha','mean'), elevation=('elevation','first'),
             latitude=('latitude','first'), longitude=('longitude','first'))
        .reset_index())

def within_cluster_r2(site, col):
    a_c = site.groupby('cluster').alpha.transform(lambda s: s - s.mean())
    x_c = site.groupby('cluster')[col].transform(lambda s: s - s.mean())
    r = np.corrcoef(a_c, x_c)[0, 1]
    return r, r**2

for col in ['elevation', 'latitude', 'longitude']:
    r, r2 = within_cluster_r2(site, col)
    print(f"{col:10s}: r = {r:+.3f}   r^2 = {r2:.3f}")

elevation : r = +0.129   r^2 = 0.017
latitude  : r = -0.141   r^2 = 0.020
longitude : r = -0.184   r^2 = 0.034


~97% of the field effect is unexplained by anything already in the panel. Elevation and
coordinates are the only static covariates present today, and together they explain a few percent at
most.

## 5. Why a *static* covariate is required

The field effect is time-invariant by construction (`alpha_hat` averages a site's own history; the
result is one number per site, not one per row). A dynamic series -- soil moisture from SMAP, or any
other time-varying remote-sensing product -- addresses the row-level residual `ε_it`, which is a
*different* question: E02-E05 already suggest that residual is not learnable at four spatial units.
Explaining a constant requires a covariate that does not move. Static soil properties (SoilGrids: clay,
sand, silt, organic carbon, pH, bulk density, depth to bedrock) are the only candidate of that shape
this repository has not yet tried -- that is E06, currently blocked on a SoilGrids API timeout.

## Sources

Full citation list, with the verified/unverified split, lives in [`docs/CONCEPTS.md`](../docs/CONCEPTS.md);
this section points at what this notebook uses and does not duplicate the entries.

- **ICC / variance components** (§2): method-of-moments estimation, Searle, Casella & McCulloch —
  **[UNVERIFIED]** —
  [`docs/CONCEPTS.md` § ICC and variance components](../docs/CONCEPTS.md#icc-and-variance-components).
- **Bootstrap confidence intervals** (§2): Efron (1979) — **[UNVERIFIED]** —
  [`docs/CONCEPTS.md` § Bootstrap confidence intervals](../docs/CONCEPTS.md#bootstrap-confidence-intervals).
- **The within / demeaning transformation** (§4): Frisch & Waugh (1933), *Econometrica* 1(4):387–401;
  Lovell (1963), *JASA* 58(304):993–1010 — **[UNVERIFIED]** —
  [`docs/CONCEPTS.md` § The within transformation, and Frisch–Waugh–Lovell](../docs/CONCEPTS.md#the-within-transformation-and-frischwaughlovell).
  Cross-sectional demeaning applied to crop-yield anomalies, *J. Agric. Biol. Environ. Stat.* (2026),
  [doi:10.1007/s13253-026-00743-8](https://doi.org/10.1007/s13253-026-00743-8) — **verified DOI;
  authorship [UNVERIFIED]**, cited by title and DOI deliberately rather than guessing the authors.

**Not citable, by design.** The field-effect decomposition itself (§2–§3: the 0.345 ICC, the
exact-zero cluster and cohort-date ICCs, the `alpha_hat` per-site estimates) and the "~97% unexplained"
finding (§4) are **measurements from this repository** — nobody published these numbers, so a
citation would be wrong. Their warrant is reproducibility: the data manifest hash, git SHA and seed
recorded with `experiments/E01_variance_decomposition.out`. See
[`docs/CONCEPTS.md`'s opening section, "On warrants, and what 'no source' means"](../docs/CONCEPTS.md#on-warrants-and-what-no-source-means).